# Parsing one page: Docling, PaddleOCR-VL and Qwen VL

Notebook demo of the AI2Future workshop *Can We Rely on Extracted Documents? Parsing Methods Compared*
(Filip Pavičić, Atomic Intelligence).

One page of a synthetic lab report goes through a document-parsing pipeline: the regions a parser finds,
the Markdown it writes, the errors it makes and the checks that catch some of them. Docling runs here, on
the CPU. PaddleOCR-VL and Qwen VL need a GPU, so their outputs on the same page are loaded from
`data/cached/`.

Sections 5 and 7 can be skipped; the other sections do not depend on them.

## Setup

On Colab, this cell clones the repository and installs the dependencies, which takes a few minutes.
Locally, install as described in the README and start Jupyter from the repository.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and Path.cwd().name != "document-understanding-demo":
    if not Path("document-understanding-demo").exists():
        !git clone --depth 1 https://github.com/Atomic-Intelligence/document-understanding-demo.git
    %cd document-understanding-demo
    !pip install --quiet -e .

# The repository root, whether Jupyter was started in notebooks/ or in the root
ROOT = Path.cwd() if (Path.cwd() / "src" / "dudemo").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
# TODO: import the helpers from dudemo, and load Docling's models once so that section 2 does not
# wait for downloads

## 1. The page

A synthetic Croatian lab report from our page generator: patient fields as key-value pairs, tables of
results with units and reference ranges, and a flag column. The page is an image, so there is no text
layer to copy: every parser has to read the pixels.

In [ ]:
# TODO: load the page and show it next to its ground-truth regions

## 2. Parse it with Docling

Docling finds the regions on the page with a layout model, reads their text with OCR and rebuilds each
table with a table-structure model. On the left are the regions it found, coloured by type; on the right
is the Markdown it writes. The time is measured on this machine.

In [ ]:
# TODO: run Docling on the page and print how long it took (the notebook's one computed number)

In [ ]:
# TODO: show_boxes(page, "docling") next to show_markdown(page, "docling")

## 3. The same page, three other parsers

PaddleOCR-VL finds the regions first and then reads each one with a small vision-language model. Qwen VL
reads the whole page in one pass and writes the regions and their text together; it is shown as released
and after our fine-tuning. Their outputs were made on a GPU and are loaded here.

In [ ]:
# TODO: the same two views for paddleocr_vl, qwen_base and qwen_finetuned

## 4. Three kinds of error

Each example shows the page above and what a parser wrote below it.

- **Visibly broken**: anyone reading the output sees that it is damaged.
- **Structure wrong**: the text may be right, but a table or a key-value pair is missing or misshapen.
- **Passes as correct**: a wrong value that looks right.

In [ ]:
# TODO: show_error for a unit written as LaTeX (PaddleOCR-VL), a table left out (general Qwen VL)
# and a reversed sign in a reference limit (Docling)

## 5. General and fine-tuned Qwen VL

The same model with the same prompt, before and after fine-tuning on generated pages. Regions are
coloured by type; a region of the page the model missed is outlined in dashed red.

*Can be skipped.*

In [ ]:
# TODO: show_boxes for qwen_base and qwen_finetuned side by side, on the second report

## 6. Checks after extraction

Simple rules that read only the output: is the unit a known unit, is the reference range shaped like a
range, do the value, its range and its flag agree. The last column compares each value with a second
parser, PaddleOCR-VL. A wrong value with the right shape passes every rule; only the second parser
disagrees with it.

In [ ]:
# TODO: run the checks on Docling's results table and show one row per line, with a mark per check

## 7. A table across pages

Parsers read one page at a time. Here the report is cut through a results table to make a second page,
and Docling reads that page without the table's header. Stacking both parts into one image and reading it
again brings the header back with the rows that continue the table.

*Can be skipped.*

In [ ]:
# TODO: cut the page through the table, run Docling on the lower part, then on both parts stacked

## 8. Try your own document

Set `MY_FILE` to a PDF or an image and run the cell. On Colab, upload the file with the file browser on
the left first.

- Docling: https://github.com/docling-project/docling
- PaddleOCR-VL: https://huggingface.co/PaddlePaddle/PaddleOCR-VL-1.6
- Qwen2.5-VL: https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct

In [ ]:
MY_FILE = "path/to/your/document.pdf"

# TODO: run Docling on MY_FILE and show the regions and the Markdown